Step 1: Project Initialization and Dependency Setup

In [ ]:
!pip install groq

In [ ]:
!pip install chromadb sentence-transformers

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Step 2: Data Loading and Preprocessing

In [ ]:
import pandas as pd

rag_df = pd.read_csv("/content/drive/MyDrive/Log_Anamoly_Detection/results/rag_documents.csv")
documents = rag_df["RAG_Document"].tolist()

print("Total RAG documents:", len(documents))
print(documents[0])

Step 3: Semantic Embedding Generation

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(documents)

print("Embedding shape:", embeddings.shape)

Step 4: Vector Database Indexing with FAISS

In [ ]:
import chromadb
from chromadb.config import Settings

client = chromadb.Client(
    Settings(
        persist_directory="./chroma_db"  # folder where DB will be saved
    )
)

collection = client.get_or_create_collection(name="anomaly_logs")

In [ ]:
if collection.count() == 0:
    embeddings = model.encode(documents)

    collection.add(
        documents=documents,
        embeddings=embeddings.tolist(),
        ids=[str(i) for i in range(len(documents))]
    )

print("Vectors stored in ChromaDB:", collection.count())

Step 5: Retrieval and LLM Analysis Logic

In [ ]:
def retrieve_similar_anomalies(query, k=3):

    query_embedding = model.encode([query]).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=k
    )

    retrieved_docs = []

    for doc, dist in zip(results["documents"][0], results["distances"][0]):
        retrieved_docs.append({
            "document": doc,
            "distance": dist  # lower = more similar
        })

    return retrieved_docs

In [ ]:
from groq import Groq

client = Groq(api_key="your_api_key")

In [ ]:
def generate_root_cause(context):
    SYSTEM_PROMPT = """You are an expert in Hadoop Distributed File System (HDFS) operations and log analysis.

You will be given retrieved anomaly cases. Each case has five sections:
- Details: raw event data for the block
- Summary: what type of anomaly occurred
- Root Cause: why it likely occurred
- Important: severity and impact notes

Use these sections as evidence. Be specific — reference the actual event sequences and counts."""

    USER_PROMPT = f"""The following anomaly cases were retrieved as similar to the query:

{context}

Based on the evidence above, provide:
1. Most likely root cause (cite specific events or patterns from the cases)
2. Anomaly category: system / network / configuration
3. Confidence level: high / medium / low — and why
4. Mitigation steps (ordered by priority)"""

    try:
        response = client.chat.completions.create(
            model="llama-3.1-8b-instant",
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": USER_PROMPT}
            ]
        )
        return response.choices[0].message.content

    except Exception as e:
        return f"[ERROR] Groq API call failed: {e}"

Step 6: End-to-End Pipeline Integration & Testing

In [ ]:
#query = "Repeated block reception with high latency"
query = """
Repeated block reception events with missing acknowledgment and high latency
"""
#query = documents[0]

similar_docs = retrieve_similar_anomalies(query)

context = "\n\n".join([
    f"Case {i+1} (distance: {r['distance']:.2f}):\n{r['document']}"
    for i, r in enumerate(similar_docs)
])

In [ ]:
result = generate_root_cause(context)

print(result)